In [16]:
import pathlib as pl
import pandas as pd


In [17]:
root_path = pl.Path.cwd().parent.parent
data_path = root_path / "data" / "raw" 

In [18]:
main_data = pd.read_csv(data_path / "googleplaystore.csv")
review_data = pd.read_csv(data_path / "googleplaystore_user_reviews.csv")

In [ ]:
main_data = main_data.drop_duplicates()
main_data["Installs"] = main_data["Installs"].str.replace(",", "").str.replace("+", "")
main_data["Installs"] = pd.to_numeric(main_data["Installs"], errors="coerce")
main_data["Installs"].dtype

main_data_exploded = main_data.copy()

main_data_exploded['Genres'] = main_data_exploded['Genres'].astype(str).str.split(';')


main_data_exploded = main_data_exploded.explode('Genres')
main_data_exploded['Rating'] = pd.to_numeric(main_data_exploded['Rating'], errors='coerce')
main_data_exploded['Last Updated'] = pd.to_datetime(main_data_exploded['Last Updated'], errors='coerce')

main_data_exploded = main_data_exploded.dropna(subset=['Installs', 'Rating', 'Last Updated'])
main_data_exploded_mais_recente = main_data_exploded['Last Updated'].max()
main_data_exploded['Days_Since_Update'] = (main_data_exploded_mais_recente - main_data_exploded['Last Updated']).dt.days
main_data_exploded = main_data_exploded[main_data_exploded.Price != "Everyone"]
main_data_exploded.Price = main_data_exploded.Price.str.replace("$","")
main_data_exploded["Price"] = pd.to_numeric(main_data_exploded["Price"],errors="coerce")
main_data_exploded.to_csv(root_path / "data" / "intermed"/ "googleplaystore_clean.csv")

In [21]:
main_data_exploded.info()

<class 'pandas.DataFrame'>
Index: 9339 entries, 0 to 10840
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   App                9339 non-null   str           
 1   Category           9339 non-null   str           
 2   Rating             9339 non-null   float64       
 3   Reviews            9339 non-null   str           
 4   Size               9339 non-null   str           
 5   Installs           9339 non-null   float64       
 6   Type               9339 non-null   str           
 7   Price              9339 non-null   float64       
 8   Content Rating     9339 non-null   str           
 9   Genres             9339 non-null   str           
 10  Last Updated       9339 non-null   datetime64[us]
 11  Current Ver        9335 non-null   str           
 12  Android Ver        9337 non-null   str           
 13  Days_Since_Update  9339 non-null   int64         
dtypes: datetime64[us](1), f

In [ ]:
review_data = review_data.dropna(subset=["Translated_Review", "Sentiment_Polarity"])
review_data = review_data.drop_duplicates(subset=["App", "Translated_Review"], keep="first")
review_data['Sentiment_Polarity'] = pd.to_numeric(review_data['Sentiment_Polarity'], errors='coerce')
review_data.to_csv(root_path / "data" / "intermed" / "googleplaystore_user_reviews_clean.csv", index=False)